# LFW 00. Protocol and run freeze

목표: 데이터 분할 누수를 검사하고 open-set protocol을 고정한 뒤, 날짜/회차별 `RunStore`를 생성합니다. 성공 기준은 protocol count와 config hash가 run artifact에 기록되는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 마십시오. 항상 **Kernel Restart 후 Run All**로 위에서부터 실행합니다. 이 노트북이 중단되면 부분 생성된 run을 재사용하지 말고 00 전체를 다시 실행해 새 run을 만드십시오. 이후 노트북은 출력된 `RUN_DIR`만 사용하며, config hash나 protocol 입력이 달라졌다면 00부터 새 run을 시작합니다. `COMPLETED`가 있는 run은 수정하지 않습니다.


In [1]:
import sys
import cv2
import insightface
import onnxruntime

print(sys.executable)
print(cv2.__version__)
print(onnxruntime.__version__)



print(onnxruntime.get_available_providers())



c:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe
5.0.0
1.20.1
['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [2]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

import pandas as pd
import yaml
from research.protocols import build_open_set_protocol, validate_identity_disjoint_splits
from research.runtime import RunStore
from research.runtime.hashing import canonical_sha256

#EXECUTE_STAGE = False  # Review inputs first; change to True for the real run.
EXECUTE_STAGE = True  # Review inputs first; change to True for the real run.
DEFAULT_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experiments' / 'lfw_face_search.yaml'
LEGACY_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experiments' / 'face_search.yaml'
CONFIG_PATH = Path(
    os.environ.get('RONBUN_LFW_CONFIG_PATH', str(DEFAULT_CONFIG_PATH))
).resolve()
if not CONFIG_PATH.is_file() and 'RONBUN_LFW_CONFIG_PATH' not in os.environ:
    CONFIG_PATH = LEGACY_CONFIG_PATH
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
CONFIG_PATH


WindowsPath('D:/ronbun/configs/experiments/lfw_face_search.yaml')

## Plan

- Validate identity-disjoint development/calibration/test splits.
- Build registered, known-unknown, and unknown-unknown probe groups separately.
- Freeze the sanitized YAML config, protocol counts, and source paths in one run.


In [3]:
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) or {}
required_sections = {'run', 'dataset', 'protocol', 'compression', 'search', 'calibration', 'certification'}
missing_sections = sorted(required_sections.difference(config))
if missing_sections:
    raise ValueError(f'Missing config sections: {missing_sections}')

dataset_cfg = config['dataset']
protocol_cfg = config['protocol']
input_paths = {
    'manifest': PROJECT_ROOT / dataset_cfg['manifest_path'],
    'gallery_identities': PROJECT_ROOT / protocol_cfg['gallery_identities_path'],
    'unknown_unknown_identities': PROJECT_ROOT / protocol_cfg['unknown_unknown_identities_path'],
}
preflight = {
    'execute_stage': EXECUTE_STAGE,
    'config_hash': canonical_sha256(config),
    'inputs_exist': {name: path.is_file() for name, path in input_paths.items()},
}
preflight


{'execute_stage': True,
 'config_hash': 'c305da878c6bcc60f36ca4640f9004726cab393851502c3e468e9db65e27de6d',
 'inputs_exist': {'manifest': True,
  'gallery_identities': True,
  'unknown_unknown_identities': True}}

## Execute and record

`EXECUTE_STAGE=False`에서는 부작용 없이 preflight만 수행합니다. 실제 실행 전에 세 입력 파일과 protocol 설정을 확인하십시오.


In [4]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    absent = [name for name, exists in preflight['inputs_exist'].items() if not exists]
    if absent:
        raise FileNotFoundError(f'Missing protocol inputs: {absent}')
    manifest = pd.read_csv(input_paths['manifest'])
    validate_identity_disjoint_splits(manifest)
    read_ids = lambda path: [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    enrollment_count = int(protocol_cfg.get('enrollment_count', protocol_cfg.get('enrollment_counts', [1])[0]))
    protocol = build_open_set_protocol(
        manifest,
        gallery_identities=read_ids(input_paths['gallery_identities']),
        unknown_unknown_identities=read_ids(input_paths['unknown_unknown_identities']),
        enrollment_count=enrollment_count,
        seed=int(protocol_cfg.get('split_seed', 42)),
    )
    counts = {
        'gallery': len(protocol.gallery),
        'registered_probes': len(protocol.registered_probes),
        'known_unknown_probes': len(protocol.known_unknown_probes),
        'unknown_unknown_probes': len(protocol.unknown_unknown_probes),
    }
    run = RunStore.create(
        experiment_name=str(config['run']['name']),
        config=config,
        root=RUN_ROOT,
        repo_root=PROJECT_ROOT,
    )
    run.record_input(CONFIG_PATH, role='experiment_config')
    for role, path in input_paths.items():
        run.record_input(path, role=role)
    with run.phase('00_protocol_and_run_freeze') as phase:
        suffix = f'A{phase.attempt:03d}'
        snapshot = {'config_hash': run.config_hash, 'inputs': {k: str(v) for k, v in input_paths.items()}, 'counts': counts}
        source = phase.attempt_dir / f'protocol_snapshot_{suffix}.json'
        source.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding='utf-8')
        phase.publish_artifact(source)
        frozen_frames = {
            'gallery': protocol.gallery,
            'registered_probes': protocol.registered_probes,
            'known_unknown_probes': protocol.known_unknown_probes,
            'unknown_unknown_probes': protocol.unknown_unknown_probes,
        }
        for name, frame in frozen_frames.items():
            csv_source = phase.attempt_dir / f'{name}_{suffix}.csv'
            frame.to_csv(csv_source, index=False)
            phase.publish_artifact(csv_source)
        phase.record_counts(**counts)
    result = {'status': 'completed', 'run_id': run.run_id, 'run_dir': str(run.run_dir), 'active_run_pointer': str(RUN_ROOT / 'active_run.json'), 'config_hash': run.config_hash, **counts}
result


{'status': 'completed',
 'run_id': '20260715-R001-c305da87',
 'run_dir': 'D:\\ronbun\\runs\\lfw\\2026\\07\\15\\20260715-R001-c305da87_thesis3_lfw_face_search_v1',
 'active_run_pointer': 'D:\\ronbun\\runs\\lfw\\active_run.json',
 'config_hash': 'c305da878c6bcc60f36ca4640f9004726cab393851502c3e468e9db65e27de6d',
 'gallery': 50,
 'registered_probes': 677,
 'known_unknown_probes': 860,
 'unknown_unknown_probes': 834}

## Next step

00이 `runs/lfw/active_run.json`에 현재 LFW run을 기록하므로 01~05는 같은 run을 자동으로 선택합니다. `RONBUN_RUN_DIR`는 여러 활성 run 중 하나를 명시적으로 고를 때만 override로 사용합니다. 00의 phase manifest가 `completed`인지 먼저 확인하십시오.


새 LFW run은 기본적으로 `configs/experiments/lfw_face_search.yaml`을 사용합니다. 파일이 아직 없으면 기존 run 호환성을 위해 `face_search.yaml`로 폴백하며, `RONBUN_LFW_CONFIG_PATH`로 명시적인 설정 파일을 지정할 수 있습니다.

LFW 전용 run pointer는 `runs/lfw/active_run.json`에 기록됩니다. 01~05는 이 위치를 먼저 확인하고, 기존 00/01 실행을 이어갈 때만 `runs/active_run.json`로 폴백합니다.